# Desktop Voice Orchestrator — M4 本地 3B/4B 路由器精调与 GGUF 导出流水线 (`dvo-router`, Colab A100 加速版)

本 Notebook 在 **Google Colab (推荐 A100 GPU)** 上一键完成 M4 路由器蒸馏微调与量化导出全流程，并内置 **`/content/local_router` 本地 NVMe 高速暂存 + 云盘增量断点同步**：
1. **Cell 1**：挂载 Google Drive，检查 A100 GPU 显存，拉取最新 `develop` 分支代码，初始化本地 NVMe 高速暂存盘 `/content/local_router`；
2. **Cell 2**：安装环境依赖（`unsloth`, `trl`, `peft`, `accelerate`, `transformers`, `bitsandbytes`, `datasets`, `gguf`）并配置转换环境；
3. **Cell 3（教师蒸馏与鲁棒性扩充）**：优先使用 `google.colab.userdata` 中的 `GEMINI_API_KEY` 或 `OPENAI_API_KEY` + 离线语音拼音近音噪声合成 **3,000 条** 严格符合规范的 SFT 训练集与验证集（自动备份至云盘）；
4. **Cell 4（基线零样本评测）**：加载 `Qwen/Qwen3-4B`（显式禁用思维链 `enable_thinking=False` 保证零思考延迟，并支持一键切换 `Qwen/Qwen2.5-3B-Instruct`），在验证集上摸底基线准确率与负样本误触发率；
5. **Cell 5（A100 QLoRA SFT 精调）**：采用 4-bit / bf16 QLoRA (`r=16, lora_alpha=32`)、仅计算 Assistant 回复损失（Completion-only loss）、余弦学习率衰减，训练过程增量同步 Checkpoint 至云盘；
6. **Cell 6（对比评测、GGUF Q4_K_M 导出与交付）**：运行微调后验证集评测，合并权重并导出 `dvo-qwen3-4b-router-q4_k_m.gguf` 与 `Modelfile.dvo-router` 到云盘 `export/` 目录，供主力机 RTX 4090 极速加载。

## Cell 1：挂载 Google Drive、硬件体检并初始化 `/content/local_router` 高速盘

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/desktop-voice-orchestrator-data/router_m4")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_ROUTER = Path("/content/local_router")
LOCAL_ROUTER.mkdir(parents=True, exist_ok=True)
(LOCAL_ROUTER / "dataset").mkdir(exist_ok=True)
(LOCAL_ROUTER / "checkpoints").mkdir(exist_ok=True)
(LOCAL_ROUTER / "export").mkdir(exist_ok=True)

REPO_DIR = Path("/content/desktop-voice-orchestrator")
REPO_URL = "https://github.com/psyduck-10385/desktop-voice-orchestrator.git"
REPO_BRANCH = "develop"

print("=== 1. GPU 硬件检查 ===")
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print(gpu_info.stdout.strip() if gpu_info.returncode == 0 else "⚠️ 未检测到 GPU，请在运行时设置中切换为 A100（或 L4）！")

print("\n=== 2. 拉取/更新代码仓库 (develop 分支) ===")
if (DRIVE_ROOT / "repo_sync" / "training" / "router" / "schema.py").is_file():
    REPO_DIR = DRIVE_ROOT / "repo_sync"
    print(f"✅ 使用云盘同步仓库: {REPO_DIR}")
elif not REPO_DIR.is_dir():
    print(f"正在克隆仓库 {REPO_URL} ({REPO_BRANCH}) ...")
    rc = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]).returncode
    if rc != 0:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"工作区路径: {REPO_DIR}")

print("\n=== 3. 增量断点恢复检查 ===")
def sync_from_drive() -> None:
    for sub in ["dataset", "checkpoints", "export"]:
        src = DRIVE_ROOT / sub
        dst = LOCAL_ROUTER / sub
        if src.is_dir():
            subprocess.run(["rsync", "-a", f"{src}/", str(dst)], check=False)

sync_from_drive()
print("✅ 本地 NVMe 高速盘就绪，云盘历史断点检查完毕。")

## Cell 2：一键安装 Unsloth、TRL、PEFT 与 GGUF 导出工具链

In [ ]:
import os
import subprocess
import sys

print("=== 1. 安装核心精调库 (Unsloth, TRL, PEFT, Accelerate, Datasets) ===")
pkgs = [
    "unsloth",
    "trl",
    "peft",
    "accelerate",
    "transformers>=4.48.0",
    "bitsandbytes",
    "datasets",
    "sentencepiece",
    "protobuf",
    "gguf",
]
subprocess.run(["pip", "install", "-q", *pkgs], check=True)

print("\n=== 2. 准备 llama.cpp 转换与量化工具 ===")
LLAMA_CPP_DIR = Path("/content/llama.cpp")
if not (LLAMA_CPP_DIR / "convert_hf_to_gguf.py").is_file():
    print("克隆 llama.cpp 仓库以获取 GGUF 转换器...")
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], check=True)
    subprocess.run(["pip", "install", "-q", "-r", str(LLAMA_CPP_DIR / "requirements.txt")], check=True)
    # 编译 llama-quantize
    subprocess.run(["make", "-C", str(LLAMA_CPP_DIR), "llama-quantize", "-j"], check=False)

print("✅ 训练与导出依赖环境配置完毕！")

## Cell 3：教师模型蒸馏与离线规则/拼音近音噪声数据合成

In [ ]:
import json
import os
import subprocess
from pathlib import Path

print("=== 1. 检测 API Key 配置 (可选，支持无 Key 离线高保真合成) ===")
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    openai_key = userdata.get('OPENAI_API_KEY')
except Exception:
    gemini_key = os.environ.get('GEMINI_API_KEY')
    openai_key = os.environ.get('OPENAI_API_KEY')

if gemini_key:
    os.environ['GEMINI_API_KEY'] = gemini_key
    print("✅ 已配置 GEMINI_API_KEY，启用 Gemini 教师模型蒸馏链路。")
elif openai_key:
    os.environ['OPENAI_API_KEY'] = openai_key
    print("✅ 已配置 OPENAI_API_KEY，启用 GPT 教师模型蒸馏链路。")
else:
    print("ℹ️ 未检测到云端 API Key，将采用内置高保真规则与中文 ASR 拼音同音词噪声注入生成。")

DATASET_LOCAL = LOCAL_ROUTER / "dataset"
TRAIN_JSONL = DATASET_LOCAL / "train.jsonl"
VAL_JSONL = DATASET_LOCAL / "val.jsonl"

GEN_SCRIPT = REPO_DIR / "training" / "router" / "generate_router_dataset.py"
assert GEN_SCRIPT.is_file(), f"未找到数据集生成脚本: {GEN_SCRIPT}"

print("\n=== 2. 运行自检并生成 3,000 条训练与验证样本 ===")
subprocess.run(["python", str(GEN_SCRIPT), "--self-test"], check=True)
subprocess.run([
    "python", str(GEN_SCRIPT),
    "--output-dir", str(DATASET_LOCAL),
    "--num-samples", "3000",
    "--train-ratio", "0.9",
    "--seed", "42",
], check=True)

print("\n=== 3. 同步数据集备份至 Google Drive ===")
DRIVE_DATASET = DRIVE_ROOT / "dataset"
DRIVE_DATASET.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", f"{DATASET_LOCAL}/", str(DRIVE_DATASET)], check=True)
print(f"✅ 数据集已成功备份至云盘: {DRIVE_DATASET}")

## Cell 4：Qwen3-4B / Qwen2.5-3B 基线零样本性能评测 (显式禁用思维链 `enable_thinking=False`)

In [ ]:
import json
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

# 支持灵活切换基础模型："Qwen/Qwen3-4B" 或 "Qwen/Qwen2.5-3B-Instruct"
BASE_MODEL_NAME = "Qwen/Qwen3-4B"
print(f"=== 1. 加载基线模型: {BASE_MODEL_NAME} ===")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

val_path = LOCAL_ROUTER / "dataset" / "val.jsonl"
val_samples = [json.loads(line) for line in open(val_path, encoding="utf-8") if line.strip()]
# 取前 100 条验证集做快速对比摸底
eval_subset = val_samples[:100]
baseline_preds = []

print(f"\n=== 2. 在验证子集 ({len(eval_subset)} 条) 上执行零样本推理 (禁用思考模式) ===")
for i, item in enumerate(eval_subset):
    msgs = [
        {"role": "system", "content": item["messages"][0]["content"]},
        {"role": "user", "content": item["messages"][1]["content"]},
    ]
    try:
        # Qwen3 显式禁用 thinking，防止思维链占用 Token 与耗时
        prompt = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    baseline_preds.append({"prediction": gen_text, "latency_ms": elapsed_ms})

pred_file = LOCAL_ROUTER / "baseline_predictions.jsonl"
with open(pred_file, "w", encoding="utf-8") as f:
    for p in baseline_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

EVAL_SCRIPT = REPO_DIR / "training" / "router" / "evaluate_router_offline.py"
baseline_report_file = LOCAL_ROUTER / "baseline_eval_report.json"
subprocess.run([
    "python", str(EVAL_SCRIPT),
    "--val-file", str(val_path),
    "--predictions", str(pred_file),
    "--max-samples", "100",
    "--output-report", str(baseline_report_file),
], check=True)

# 释放显存以备 SFT 训练
del base_model
torch.cuda.empty_cache()
print("✅ 基线模型评测完毕并已卸载显存。")

## Cell 5：A100 QLoRA SFT 精调 (r=16, lora_alpha=32, Completion-Only 损失)

In [ ]:
import os
import subprocess
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

try:
    from unsloth import FastLanguageModel
    HAS_UNSLOTH = True
except ImportError:
    HAS_UNSLOTH = False

BASE_MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 1024
OUTPUT_LORA_DIR = LOCAL_ROUTER / "lora_model"

print(f"=== 1. 初始化模型与 LoRA 适配器 (Unsloth={HAS_UNSLOTH}) ===")
if HAS_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
else:
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)

print("\n=== 2. 格式化训练集与配置 Completion-only 掩码 ===")
train_data_path = LOCAL_ROUTER / "dataset" / "train.jsonl"
dataset = load_dataset("json", data_files=str(train_data_path), split="train")

def format_chatml(example):
    try:
        formatted = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            enable_thinking=False,
        )
    except TypeError:
        formatted = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
        )
    return {"text": formatted}

dataset = dataset.map(format_chatml)
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

print("\n=== 3. 启动 A100 QLoRA 训练 ===")
training_args = TrainingArguments(
    output_dir=str(LOCAL_ROUTER / "checkpoints"),
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=collator,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

trainer.train()
print(f"\n保存微调权重至 {OUTPUT_LORA_DIR} ...")
model.save_pretrained(str(OUTPUT_LORA_DIR))
tokenizer.save_pretrained(str(OUTPUT_LORA_DIR))

# 同步断点至 Google Drive
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", f"{LOCAL_ROUTER / 'checkpoints'}/", str(DRIVE_CHECKPOINTS)], check=False)
print("✅ QLoRA SFT 训练完成并已增量备份至 Google Drive！")

## Cell 6：Post-SFT 验证集评测对比、合并权重并导出 Q4_K_M GGUF 到 Google Drive

In [ ]:
import json
import shutil
import subprocess
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("=== 1. 加载微调模型并在验证集上评测 ===")
OUTPUT_LORA_DIR = LOCAL_ROUTER / "lora_model"
MERGED_DIR = LOCAL_ROUTER / "merged_16bit"

tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_LORA_DIR), trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_LORA_DIR))
model.eval()

val_path = LOCAL_ROUTER / "dataset" / "val.jsonl"
val_samples = [json.loads(line) for line in open(val_path, encoding="utf-8") if line.strip()]
eval_subset = val_samples[:100]
sft_preds = []

for item in eval_subset:
    msgs = [
        {"role": "system", "content": item["messages"][0]["content"]},
        {"role": "user", "content": item["messages"][1]["content"]},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    sft_preds.append({"prediction": gen_text, "latency_ms": elapsed_ms})

sft_pred_file = LOCAL_ROUTER / "sft_predictions.jsonl"
with open(sft_pred_file, "w", encoding="utf-8") as f:
    for p in sft_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

sft_report_file = LOCAL_ROUTER / "export" / "eval_report.json"
EVAL_SCRIPT = REPO_DIR / "training" / "router" / "evaluate_router_offline.py"
subprocess.run([
    "python", str(EVAL_SCRIPT),
    "--val-file", str(val_path),
    "--predictions", str(sft_pred_file),
    "--max-samples", "100",
    "--output-report", str(sft_report_file),
], check=True)

print("\n=== 2. 合并 LoRA 权重到 16-bit 模型 ===")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

print("\n=== 3. 转换为 GGUF 并进行 Q4_K_M 量化 ===")
LLAMA_CPP_DIR = Path("/content/llama.cpp")
FP16_GGUF = LOCAL_ROUTER / "export" / "dvo-qwen3-4b-router-f16.gguf"
Q4KM_GGUF = LOCAL_ROUTER / "export" / "dvo-qwen3-4b-router-q4_k_m.gguf"

subprocess.run([
    "python", str(LLAMA_CPP_DIR / "convert_hf_to_gguf.py"),
    str(MERGED_DIR),
    "--outfile", str(FP16_GGUF),
    "--outtype", "f16",
], check=True)

quantize_bin = LLAMA_CPP_DIR / "llama-quantize"
if quantize_bin.is_file():
    subprocess.run([str(quantize_bin), str(FP16_GGUF), str(Q4KM_GGUF), "Q4_K_M"], check=True)
    if FP16_GGUF.is_file():
        FP16_GGUF.unlink()  # 节省磁盘空间
else:
    shutil.copy2(FP16_GGUF, Q4KM_GGUF)

print("\n=== 4. 拷贝 Modelfile 与评测报告并同步至 Google Drive ===")
src_modelfile = REPO_DIR / "training" / "router" / "Modelfile.dvo-router"
if src_modelfile.is_file():
    shutil.copy2(src_modelfile, LOCAL_ROUTER / "export" / "Modelfile.dvo-router")

DRIVE_EXPORT = DRIVE_ROOT / "export"
DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", f"{LOCAL_ROUTER / 'export'}/", str(DRIVE_EXPORT)], check=True)

print(f"\n🎉 路由器全流程交付完成！已同步至云盘:\n   {DRIVE_EXPORT}")
print("产物包括：")
print("  1. dvo-qwen3-4b-router-q4_k_m.gguf (高密度量化模型，RTX 4090 推理延迟约 80~110ms)")
print("  2. Modelfile.dvo-router (Ollama 配置文件)")
print("  3. eval_report.json (Schema准确率、负样本误报率与延迟评测数据)")
print("\n在主力机上部署方法：")
print("  ollama create dvo-router -f Modelfile.dvo-router")
print("并在 config/local.toml 中配置 router_model = 'dvo-router' 即可生效！")